# Integrate Claude Code with MCP Server using AgentCore Gateway

## Overview

In large enterprises with thousands of development teams, each running multiple MCP servers with several tools, managing and discovering tools becomes a critical challenge:

* **Tool Discovery and Sharing**: Teams struggle to discover and consume tools across the organization. Should each team maintain separate gateways? How do you share gateway URLs and manage central registries without creating operational overhead?
* **Gateway Management**: Maintaining separate gateways for each MCP server quickly becomes unmanageable at scale.
* **Authentication Complexity**: Managing authentication and authorization across multiple MCP servers becomes increasingly complex, especially with sensitive enterprise data.
* **Maintenance Overhead**: Keeping up with MCP specification updates requires continuous rework and testing across all implementations.

AgentCore Gateway now supports onboarding pre-existing MCP server implementations as targets, in addition to Lambda and OpenAPI tools. This tutorial demonstrates the concept using a simple MCP server in AgentCore Runtime for this implementation.

![Diagram](images/mcp-server-target.png)

Once the AgentCore Gateway is integrated with the MCP server as a target, we'll search for tools through our integrated MCP server, then we'll use a Strands Agent to demonstrate the list tools and invoke tool functionality. More detail on each of these flows can be found below:

### Search tool behavior for MCP Targets

The search capability in AgentCore Gateway enables semantic discovery of tools across all target types, including MCP targets. For MCP targets, the search functionality operates on normalized tool definitions that were captured and indexed during synchronization operations, providing efficient semantic search without real-time MCP server communication. When tool definitions are synchronized from an MCP target, Gateway automatically generates embeddings for each tool's name, description, and parameter descriptions. These embeddings are stored alongside the normalized tool definitions, enabling semantic search that understands the intent and context of search queries. Unlike traditional keyword matching, this allows agents to discover relevant tools even when exact terminology doesn't match.

![Search](images/mcp-server-search-tool.png)

### ListTools behavior for MCP Targets

The ListTools operation in AgentCore Gateway provides access to tool definitions previously synchronized from MCP targets, following a cache-first approach that prioritizes performance and reliability. Unlike traditional OpenAPI or Lambda targets where tool definitions are statically defined, MCP target tools are discovered and cached through synchronization operations. When a client calls ListTools, the Gateway retrieves tool definitions from its persistent storage rather than making real-time calls to the MCP server. These definitions were previously populated either through implicit synchronization during target creation/update or through explicit SynchronizeGateway API calls. The operation returns a paginated list of normalized tool definitions.

![List](images/mcp-server-list-tools.png)

### InvokeTool (tools/call) Behavior for MCP Targets

The InvokeTool operation for MCP targets handles the actual execution of tools discovered through ListTools, managing real-time communication with the target MCP server. Unlike the cache-based ListTools operation, tools/call requires active communication with the MCP server, introducing specific authentication, session management, and error handling requirements. When a tools/call request arrives, Gateway first validates the tool exists in its synchronized definitions. For MCP targets, Gateway performs an initial 'initialize' call to establish a session with the MCP server. If the target is configured with OAuth credentials, Gateway retrieves fresh credentials from AgentCore Identity before making the initialize call. This ensures that even if ListTools returned cached tools with expired credentials, the actual invocation uses valid authentication.

![Invoke](images/mcp-server-invoke-tool.png)


### Tutorial Details


| Information          | Details                                                   |
|:---------------------|:----------------------------------------------------------|
| Tutorial type        | Interactive                                               |
| AgentCore components | AgentCore Gateway, AgentCore Identity                     |
| Gateway Target type  | MCP server                                                |
| Inbound Auth IdP     | Amazon Cognito, but can use others                        |
| LLM model            | Anthropic Claude Sonnet 4                                 |
| Tutorial components  | Creating AgentCore Gateway and Invoking AgentCore Gateway from Claude Code |
| Tutorial vertical    | Cross-vertical                                            |
| Example complexity   | Easy                                                      |
| SDK used             | boto3                                                     |

## Tutorial architecture

This tutorial serves as a practical example of the broader enterprise challenge: **How to integrate existing MCP servers into a centralized Gateway architecture.**

## Prerequisites

To execute this tutorial you will need:
* Jupyter notebook (Python kernel)
* uv
* AWS credentials
* Amazon Cognito

In [ ]:
# Install from the requirements file or pyproject.toml file in current directory
!pip install -U -r requirements.txt --quiet

In [ ]:
import os
# Set AWS credentials if not using SageMaker notebook
# os.environ['AWS_ACCESS_KEY_ID'] = '' # Set the access key
# os.environ['AWS_SECRET_ACCESS_KEY'] = '' # Set the secret key
os.environ['AWS_DEFAULT_REGION'] = os.environ.get('AWS_REGION', 'us-east-1')

In [ ]:
from dotenv import load_dotenv

load_dotenv()

print(os.environ['TAVILY_MCP_URL'].split('tvly')[0])

In [ ]:
# Import utils
import os
import sys

# Get the directory of the current script
if '__file__' in globals():
    current_dir = os.path.dirname(os.path.abspath(__file__))
else:
    current_dir = os.getcwd()  # Fallback if __file__ is not defined (e.g., Jupyter)

# Navigate to the directory containing utils.py (one level up)
utils_dir = os.path.abspath(os.path.join(current_dir, '..'))

# Add to sys.path
sys.path.insert(0, utils_dir)

# Now you can import utils
import utils

# Setup logging 
import logging

# Configure logging for notebook environment
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    handlers=[logging.StreamHandler()]
)

# Set specific logger levels
logging.getLogger("strands").setLevel(logging.INFO)


## Create an IAM role for the Gateway to assume

In [ ]:
agentcore_gateway_iam_role = utils.create_agentcore_gateway_role("sample-claude-code-mcp-gateway")
print("Agentcore gateway role ARN: ", agentcore_gateway_iam_role['Role']['Arn'])

## Create Amazon Cognito Pool for Inbound authorization to Gateway

In [ ]:
# Creating Cognito User Pool 
import os
import boto3

REGION = os.environ['AWS_DEFAULT_REGION']
USER_POOL_NAME = "sample-agentcore-gateway-pool"
RESOURCE_SERVER_ID = "sample-agentcore-gateway-id"
RESOURCE_SERVER_NAME = "sample-agentcore-gateway-name"
CLIENT_NAME = "sample-agentcore-gateway-client"
SCOPES = [
    {"ScopeName": "invoke",  # Just 'invoke', will be formatted as resource_server_id/invoke
    "ScopeDescription": "Scope for invoking the agentcore gateway"},
]

scope_names = [f"{RESOURCE_SERVER_ID}/{scope['ScopeName']}" for scope in SCOPES]
scopeString = " ".join(scope_names)


cognito = boto3.client("cognito-idp", region_name=REGION)

print("Creating or retrieving Cognito resources...")
gw_user_pool_id = utils.get_or_create_user_pool(cognito, USER_POOL_NAME)
print(f"User Pool ID: {gw_user_pool_id}")

utils.get_or_create_resource_server(cognito, gw_user_pool_id, RESOURCE_SERVER_ID, RESOURCE_SERVER_NAME, SCOPES)
print("Resource server ensured.")

gw_client_id, gw_client_secret = utils.get_or_create_m2m_client(cognito, gw_user_pool_id, CLIENT_NAME, RESOURCE_SERVER_ID, scope_names)

print(f"Client ID: {gw_client_id}")

# Get discovery URL
gw_cognito_discovery_url = f'https://cognito-idp.{REGION}.amazonaws.com/{gw_user_pool_id}/.well-known/openid-configuration'
print(gw_cognito_discovery_url)

## Create the Gateway

In [ ]:
# CreateGateway with Cognito authorizer. Use the Cognito user pool created in the previous step
import boto3
gateway_client = boto3.client('bedrock-agentcore-control', region_name=REGION)
auth_config = {
    "customJWTAuthorizer": { 
        "allowedClients": [gw_client_id],  # Client MUST match with the ClientId configured in Cognito. Example: 7rfbikfsm51j2fpaggacgng84g
        "discoveryUrl": gw_cognito_discovery_url
    }
}
create_response = gateway_client.create_gateway(name='claude-code-gateway-mcp-server',
    roleArn=agentcore_gateway_iam_role['Role']['Arn'], # The IAM Role must have permissions to create/list/get/delete Gateway
    protocolType='MCP',
    protocolConfiguration={
        'mcp': {
            'supportedVersions': ['2025-03-26'],
            'searchType': 'SEMANTIC'
        }
    },
    authorizerType='CUSTOM_JWT',
    authorizerConfiguration=auth_config,
    description='AgentCore Gateway with MCP Server target'
)
print(create_response)
# Retrieve the GatewayID used for GatewayTarget creation
gatewayID = create_response["gatewayId"]
gatewayURL = create_response["gatewayUrl"]
print(gatewayID)

In [ ]:
print(gatewayURL)

## Create an MCP server as target for AgentCore Gateway

### Create the AWS Knowledge Gateway Target

In [ ]:
import boto3

mcp_server_aws_kb_endpoint = "https://knowledge-mcp.global.api.aws"

gateway_client = boto3.client('bedrock-agentcore-control', region_name=REGION)
create_gateway_target_aws_kb_response = gateway_client.create_gateway_target(
    name='mcp-server-target-aws-kb',
    gatewayIdentifier=gatewayID,
    targetConfiguration={
        'mcp': {
            'mcpServer': {
                'endpoint': mcp_server_aws_kb_endpoint
            }
        }
    },
)
print(create_gateway_target_aws_kb_response)

### Create the Tavily Gateway Target

In [ ]:
tavily_mcp_endpoint = os.environ['TAVILY_MCP_URL']

gateway_client = boto3.client('bedrock-agentcore-control', region_name=REGION)
create_gateway_target_tavily_response = gateway_client.create_gateway_target(
    name='mcp-server-target-tavily',
    gatewayIdentifier=gatewayID,
    targetConfiguration={
        'mcp': {
            'mcpServer': {
                'endpoint': tavily_mcp_endpoint
            }
        }
    },
)
print(create_gateway_target_tavily_response)

#### Check that the Gateway targets exists, and are READY

In [ ]:
list_targets_response = gateway_client.list_gateway_targets(gatewayIdentifier=gatewayID)
print(list_targets_response)

#### Request the access token from Amazon Cognito for inbound authorization

In [ ]:
print("Requesting the access token from Amazon Cognito authorizer...May fail for some time till the domain name propagation completes")
token_response = utils.get_token(gw_user_pool_id, gw_client_id, gw_client_secret, scopeString, REGION)
token = token_response["access_token"]
print("Token response:", token)

### Add the AgentCore Gateway as MCP Server to Claude Code

In [ ]:
!claude mcp add --transport http claude-code-gateway-mcp-server {gatewayURL}  --header "Authorization: Bearer {token}"

#### Check the AgentCore Gateway shows as Connected

In [ ]:
!claude mcp list

#### Lets check current directory


In [ ]:
print(os.getcwd())

#### Listing MCP Tools in Claude Code

- Open Terminal and change directory into the current directory (from the previous cell you run). 
- Run `Claude` to open Claude Code
- Choose `claude-code-gateway-mcp-server`
- Type `/mcp` to view the MCP Tools
- Choose `View tools`
- You should see `mcp-server-target-aws-kb___aws___` prefixed tools.
- Go down with the list. You should be able to see also the `mcp-server-target-tavily___` prefixed tools.

## Clean up
Additional resources are also created like IAM role, IAM Policies, Credentials provider, AWS Lambda functions, Cognito user pools, s3 buckets that you might need to manually delete as part of the clean up. This depends on the example you run.


### Delete the Gateway (Optional)

In [ ]:
utils.delete_gateway(gateway_client, gatewayID)

### Remove the AgentCore Gateway MCP Server from Claude Code


In [ ]:
!claude mcp remove claude-code-gateway-mcp-server